# Churn Markers Analysis

Using **only 2016-2020 data**, find behavioral markers that distinguish churned vs active customers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

panel = pd.read_csv('Processed Data/big_panel.csv', parse_dates=['ORDER_DATE'])

## Step 1: Define churn labels (Advanced rule)

In [ ]:
orders_only = panel[panel['is_return_order'] == False].copy()

train_mask = (orders_only['ORDER_DATE'] >= '2016-01-01') & (orders_only['ORDER_DATE'] <= '2020-12-31')
val_mask = (orders_only['ORDER_DATE'] >= '2020-01-01') & (orders_only['ORDER_DATE'] <= '2025-12-31')

train_hids = set(orders_only[train_mask]['H_ID'].unique())
val_hids = set(orders_only[val_mask]['H_ID'].unique())
churned_hids = train_hids - val_hids

## Step 2: Build features from 2016-2020 data only

In [ ]:
train_orders = panel[(panel['ORDER_DATE'] >= '2016-01-01') & (panel['ORDER_DATE'] <= '2020-12-31')].copy()
train_purchases = train_orders[train_orders['is_return_order'] == False].copy()
train_returns = train_orders[train_orders['is_return_order'] == True].copy()

n_total_orders = train_purchases.groupby('H_ID')['ORDER_NO'].nunique().rename('n_orders')
n_return_orders = train_returns.groupby('H_ID')['ORDER_NO'].nunique().rename('n_return_orders')

return_rate = (n_return_orders / n_total_orders).fillna(0).rename('return_rate')

total_spent = train_purchases.groupby('H_ID')['net_amount_order'].sum().rename('total_spent')
avg_order_value = train_purchases.groupby('H_ID')['net_amount_order'].mean().rename('avg_order_value')

has_discount = train_purchases.groupby('H_ID')['has_discount'].any().rename('has_discount')

n_unique_items = train_purchases.groupby('H_ID')['n_unique_items'].sum().rename('n_unique_items')
avg_line_items = train_purchases.groupby('H_ID')['n_line_items'].mean().rename('avg_line_items')

first_order = train_purchases.groupby('H_ID')['ORDER_DATE'].min().rename('first_order')
last_order = train_purchases.groupby('H_ID')['ORDER_DATE'].max().rename('last_order')
days_active = ((last_order - first_order).dt.days).rename('days_active')
days_before_cutoff = (pd.Timestamp('2020-12-31') - last_order).dt.days.rename('days_before_cutoff')

n_seasons = train_purchases.groupby('H_ID')['ORDER_DATE'].apply(
    lambda x: x.dt.to_period('Q').nunique()
).rename('n_seasons_active')

orders_per_year = (n_total_orders / (days_active.clip(lower=1) / 365)).rename('orders_per_year')

features = pd.concat([
    n_total_orders, return_rate, total_spent, avg_order_value,
    has_discount, n_unique_items, avg_line_items,
    days_active, days_before_cutoff, n_seasons, orders_per_year
], axis=1).reset_index()

features['churned'] = features['H_ID'].isin(churned_hids)

features.to_csv('Processed Data/churn_features.csv', index=False)

n_churned = features['churned'].sum()
n_active = (~features['churned']).sum()
print(f'Training customers: {len(features):,}')
print(f'Active: {n_active:,} ({n_active/len(features)*100:.1f}%)')
print(f'Churned: {n_churned:,} ({n_churned/len(features)*100:.1f}%)')

## Step 3: Compare churned vs active customers

In [ ]:
compare_cols = ['n_orders', 'return_rate', 'total_spent', 'avg_order_value',
                'n_unique_items', 'avg_line_items', 'days_active',
                'days_before_cutoff', 'n_seasons_active', 'orders_per_year']

active = features[~features['churned']]
churned = features[features['churned']]

print(f'{"Feature":<25} {"Active (median)":>18} {"Churned (median)":>18} {"Difference":>12}')
print('-' * 75)
for col in compare_cols:
    a = active[col].median()
    c = churned[col].median()
    diff = c - a
    print(f'{col:<25} {a:>18.1f} {c:>18.1f} {diff:>+12.1f}')

## Step 4: Visualize key markers

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

plot_features = ['n_orders', 'return_rate', 'total_spent', 'days_active', 'n_seasons_active', 'orders_per_year']

for ax, col in zip(axes.flat, plot_features):
    active[col].hist(bins=50, alpha=0.6, label='Active', ax=ax)
    churned[col].hist(bins=50, alpha=0.6, label='Churned', ax=ax)
    ax.set_title(col)
    ax.legend()

plt.suptitle('Churned vs Active: Feature Distributions (2016-2020 data)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

plot_features = ['n_orders', 'return_rate', 'total_spent', 'days_active', 'n_seasons_active', 'orders_per_year']

for ax, col in zip(axes.flat, plot_features):
    churn_rate = features.groupby(col)['churned'].mean()
    if features[col].nunique() > 20:
        churn_rate = features.groupby(pd.qcut(features[col], q=10, duplicates='drop'))['churned'].mean()
    churn_rate.plot(kind='bar', ax=ax)
    ax.set_title(f'Churn rate by {col}')
    ax.set_ylabel('Churn rate')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Churn Rate by Feature Buckets', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5: Discount impact

In [ ]:
discount_churn = features.groupby('has_discount')['churned'].agg(['mean', 'count'])
discount_churn.index = ['No Discount', 'Had Discount']
discount_churn.columns = ['Churn Rate', 'N Customers']
display(discount_churn)

## Step 6: One-time buyers vs repeat buyers

In [ ]:
features['buyer_type'] = features['n_orders'].apply(lambda x: '1 order' if x == 1 else ('2-3 orders' if x <= 3 else '4+ orders'))

buyer_churn = features.groupby('buyer_type')['churned'].agg(['mean', 'count'])
buyer_churn.columns = ['Churn Rate', 'N Customers']
display(buyer_churn)